## Emotion Detection Innovation

This notebook uses the final RAF-DB transfer learning model for real-time emotion detection from webcam input.

The main improvement here is not another training run. The model already performs well on the RAF-DB test set, but webcam prediction can still be unstable because lighting, face angle, movement, and small expression changes affect the input.

To make the system more useful in real-time, I added confidence checking, temporal smoothing, tighter face cropping, and a live session summary. Instead of trusting one frame at a time, the system uses recent prediction probabilities to make the displayed emotion more stable.

In [32]:
import cv2
import time
import torch
import torch.nn as nn

from PIL import Image
from collections import deque, Counter
from torchvision import models, transforms

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [34]:
class_names = {
    0: "surprise",
    1: "fear",
    2: "disgust",
    3: "happy",
    4: "sad",
    5: "angry",
    6: "neutral"
}

num_classes = len(class_names)

print(class_names)

{0: 'surprise', 1: 'fear', 2: 'disgust', 3: 'happy', 4: 'sad', 5: 'angry', 6: 'neutral'}


In [35]:
model = models.resnet34(pretrained=False)

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, num_classes)
)

model.load_state_dict(
    torch.load("../models/emotion_model_rafdbv1.pth", map_location=device)
)

model = model.to(device)
model.eval()

print("Final RAF-DB emotion model loaded successfully")

Final RAF-DB emotion model loaded successfully


In [36]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [37]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    print("Face detector could not be loaded")
else:
    print("Face detector loaded")

Face detector loaded


In [38]:
def crop_face(frame, face_box):
    x, y, w, h = face_box


    padding_x = int(w * 0.08)
    padding_y_top = int(h * 0.05)
    padding_y_bottom = int(h * 0.08)

    x1 = max(x + padding_x, 0)
    y1 = max(y + padding_y_top, 0)
    x2 = min(x + w - padding_x, frame.shape[1])
    y2 = min(y + h - padding_y_bottom, frame.shape[0])

    face_crop = frame[y1:y2, x1:x2]

    return face_crop, (x1, y1, x2, y2)

In [39]:
def predict_face_emotion(face_crop):
    rgb_face = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    pil_face = Image.fromarray(rgb_face)

    input_tensor = transform(pil_face).unsqueeze(0)
    input_tensor = input_tensor.to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1).squeeze(0)

    return probabilities.cpu()

In [40]:
def draw_probability_bars(frame, avg_probs, start_x=20, start_y=90):
    bar_width = 160
    bar_height = 18
    gap = 8

    for i in range(num_classes):
        emotion = class_names[i]
        prob = float(avg_probs[i])

        y = start_y + i * (bar_height + gap)

        cv2.putText(
            frame,
            f"{emotion}: {prob:.2f}",
            (start_x, y - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1
        )

        cv2.rectangle(
            frame,
            (start_x, y),
            (start_x + bar_width, y + bar_height),
            (80, 80, 80),
            1
        )

        cv2.rectangle(
            frame,
            (start_x, y),
            (start_x + int(bar_width * prob), y + bar_height),
            (0, 255, 0),
            -1
        )

In [41]:
# Main innovation settings
confidence_threshold = 0.45
smoothing_window = 8

probability_history = deque(maxlen=smoothing_window)
stable_prediction_history = deque(maxlen=30)
session_counts = Counter()

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Webcam not found")
else:
    print("Webcam started")
    print("Press 'q' to quit")
    print("Press 'r' to reset session summary")

prev_time = time.time()

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to read frame")
        break

    frame = cv2.flip(frame, 1)

    current_time = time.time()
    fps = 1 / (current_time - prev_time)
    prev_time = current_time

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(80, 80)
    )

    display_text = "No face detected"
    display_color = (0, 0, 255)

    if len(faces) > 0:
        largest_face = max(faces, key=lambda box: box[2] * box[3])

        face_crop, crop_box = crop_face(frame, largest_face)
        x1, y1, x2, y2 = crop_box

        if face_crop.size > 0:
            probs = predict_face_emotion(face_crop)
            probability_history.append(probs)

            avg_probs = torch.stack(list(probability_history)).mean(dim=0)

            confidence, predicted_class = torch.max(avg_probs, dim=0)

            confidence_score = float(confidence)
            predicted_index = int(predicted_class)
            predicted_emotion = class_names[predicted_index]

            if confidence_score < confidence_threshold:
                final_emotion = "uncertain"
                display_color = (0, 255, 255)
            else:
                final_emotion = predicted_emotion
                display_color = (0, 255, 0)
                session_counts[final_emotion] += 1
                stable_prediction_history.append(final_emotion)

            display_text = f"{final_emotion} ({confidence_score:.2f})"

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                display_color,
                2
            )

            cv2.putText(
                frame,
                display_text,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                display_color,
                2
            )

            draw_probability_bars(frame, avg_probs)

    else:
        probability_history.clear()

    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Threshold: {confidence_threshold}",
        (20, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        1
    )

    # Session summary on right side
    summary_x = frame.shape[1] - 230
    summary_y = 30

    cv2.putText(
        frame,
        "Session summary",
        (summary_x, summary_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    for idx, emotion in enumerate(class_names.values()):
        count = session_counts[emotion]
        cv2.putText(
            frame,
            f"{emotion}: {count}",
            (summary_x, summary_y + 30 + idx * 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            1
        )

    cv2.putText(
        frame,
        "q: quit | r: reset",
        (20, frame.shape[0] - 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        1
    )

    cv2.imshow("Emotion Detection Innovation", frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q"):
        break

    if key == ord("r"):
        session_counts.clear()
        stable_prediction_history.clear()
        probability_history.clear()
        print("Session summary reset")

cap.release()
cv2.destroyAllWindows()

print("Webcam closed")
print("Final session emotion counts:")
print(session_counts)

Webcam started
Press 'q' to quit
Press 'r' to reset session summary
Webcam closed
Final session emotion counts:
Counter({'neutral': 361, 'fear': 119, 'disgust': 102, 'angry': 52, 'happy': 35, 'surprise': 22, 'sad': 8})


### What this innovation adds

The normal model predicts emotion frame by frame. This can be unstable because a single webcam frame may be affected by face movement, lighting, blur, or a weak expression.

This version improves the real-time system in four ways:

1. It detects the face and crops the face area before prediction, so the model focuses on the face instead of the full webcam frame.

2. It uses a tighter crop around the face so that extra background and head area are reduced.

3. It applies confidence checking. If the model confidence is too low, the system shows "uncertain" instead of forcing an emotion prediction.

4. It uses temporal smoothing. Instead of relying on one frame, it averages prediction probabilities across recent frames. This makes the displayed emotion more stable.

The notebook also keeps a simple live session summary, showing how often each emotion was detected during the webcam session.

### Why this is useful

In real-time webcam emotion recognition, predictions can change quickly between frames. This does not always mean the person's emotion is changing. It can happen because of lighting changes, small face movements, or uncertain predictions.

The confidence threshold reduces incorrect forced predictions, and temporal smoothing reduces flickering. This makes the system feel more reliable for real-world use.